# Murphy 1 · Chapter 14 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#M14). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## 14.1 · Convolution and pooling from scratch  
*Section §14.2 · code*

Write conv2d (valid, stride 1) and 2×2 max-pooling with NumPy loops. Apply a vertical-edge kernel to an image whose left half is dark and right half bright, then pool. Check the conv result against a vectorised version using sliding windows.

<details><summary><b>Hint</b></summary>

np.lib.stride_tricks.sliding_window_view(x, (3, 3)) gives every 3×3 patch.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
x = np.zeros((8, 8)); x[:, 4:] = 1.0
K = np.array([[-1, 0, 1]] * 3, dtype=float)
def conv2d(x, K):
    k = K.shape[0]; out = np.zeros((x.shape[0] - k + 1, x.shape[1] - k + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = ...  # TODO: one dot product per position
    return out
def maxpool(x):
    H, W = x.shape[0] // 2, x.shape[1] // 2
    return ...  # TODO: max over each 2x2 block
y = conv2d(x, K)
fast = np.einsum("ijkl,kl->ij", np.lib.stride_tricks.sliding_window_view(x, (3, 3)), K)
print("conv output row 0:", y[0]); print("loops == vectorised:", np.allclose(y, fast))
print("after 2x2 max-pool:\n", maxpool(y))

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
x = np.zeros((8, 8)); x[:, 4:] = 1.0
K = np.array([[-1, 0, 1]] * 3, dtype=float)
def conv2d(x, K):
    k = K.shape[0]; out = np.zeros((x.shape[0] - k + 1, x.shape[1] - k + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(x[i:i + k, j:j + k] * K)  # one dot product per position
    return out
def maxpool(x):
    H, W = x.shape[0] // 2, x.shape[1] // 2
    return x[:2 * H, :2 * W].reshape(H, 2, W, 2).max(axis=(1, 3))  # max over each 2x2 block
y = conv2d(x, K)
fast = np.einsum("ijkl,kl->ij", np.lib.stride_tricks.sliding_window_view(x, (3, 3)), K)
print("conv output row 0:", y[0]); print("loops == vectorised:", np.allclose(y, fast))
print("after 2x2 max-pool:\n", maxpool(y))

<details><summary><b>Answer and explanation</b></summary>

The edge kernel responds only along the boundary column, and pooling keeps that response while halving the size. The loop and the vectorised version agree exactly: a convolution is the same small dot product at every position (Murphy §14.2.1–14.2.2).

Expected output:
```
conv output row 0: [0. 0. 3. 3. 0. 0.]
loops == vectorised: True
after 2x2 max-pool:
 [[0. 3. 0.]
 [0. 3. 0.]
 [0. 3. 0.]]
```

</details>

## 14.2 · Count the weights  
*Section §14.2 · think*

A 32×32 RGB image goes into (a) a fully connected layer with 1,024 outputs, (b) a conv layer with 64 filters of size 3×3. How many weights (ignore biases) does each have, and how many outputs?

<details><summary><b>Hint</b></summary>

Dense: inputs × outputs. Conv: k × k × in_channels × out_channels.

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

(a) 32·32·3 = 3,072 inputs × 1,024 = 3,145,728 weights, 1,024 outputs. (b) 3·3·3·64 = 1,728 weights, and with 'same' padding 32·32·64 = 65,536 outputs. The conv layer has about 1,800× fewer weights yet 64× more outputs, because the same 1,728 numbers are reused at every position (Murphy §14.2.1).

</details>

## 14.3 · Why residual connections help, measured  
*Section §14.3 · code*

Build a 50-layer ReLU network (width 64, weights with std 1/√n) with and without residual connections x ← x + 0.1·ReLU(Wx). Backpropagate a unit gradient and compare its norm at layer 1.

<details><summary><b>Hint</b></summary>

Without a skip the layer Jacobian is W·diag(mask); with one it is I + 0.1·W·diag(mask).

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(8)
n, L = 64, 50
Ws = [rng.normal(0, 1 / np.sqrt(n), (n, n)) for _ in range(L)]
x = rng.normal(size=n); xr = x.copy(); masks_p, masks_r = [], []
for W in Ws:                                       # forward: plain ReLU stack vs residual stack
    z = W @ x; masks_p.append(z > 0); x = np.maximum(z, 0)
    zr = W @ xr; masks_r.append(zr > 0); xr = xr + 0.1 * np.maximum(zr, 0)
g_p = np.ones(n) / 8; g_r = np.ones(n) / 8         # unit-norm gradient arriving from the top
for W, mp, mr in zip(reversed(Ws), reversed(masks_p), reversed(masks_r)):
    g_p = W.T @ (g_p * mp)                         # Jacobian of x -> relu(Wx)
    g_r = ...  # TODO: Jacobian of x -> x + 0.1 relu(Wx): identity plus a small term
print(f"plain ReLU stack: |grad at layer 1| = {np.linalg.norm(g_p):.3e}")
print(f"residual stack:   |grad at layer 1| = {np.linalg.norm(g_r):.3e}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(8)
n, L = 64, 50
Ws = [rng.normal(0, 1 / np.sqrt(n), (n, n)) for _ in range(L)]
x = rng.normal(size=n); xr = x.copy(); masks_p, masks_r = [], []
for W in Ws:                                       # forward: plain ReLU stack vs residual stack
    z = W @ x; masks_p.append(z > 0); x = np.maximum(z, 0)
    zr = W @ xr; masks_r.append(zr > 0); xr = xr + 0.1 * np.maximum(zr, 0)
g_p = np.ones(n) / 8; g_r = np.ones(n) / 8         # unit-norm gradient arriving from the top
for W, mp, mr in zip(reversed(Ws), reversed(masks_p), reversed(masks_r)):
    g_p = W.T @ (g_p * mp)                         # Jacobian of x -> relu(Wx)
    g_r = g_r + 0.1 * (W.T @ (g_r * mr))  # Jacobian of x -> x + 0.1 relu(Wx): identity plus a small term
print(f"plain ReLU stack: |grad at layer 1| = {np.linalg.norm(g_p):.3e}")
print(f"residual stack:   |grad at layer 1| = {np.linalg.norm(g_r):.3e}")

<details><summary><b>Answer and explanation</b></summary>

Without skips the gradient shrinks to about 10⁻⁸: each ReLU layer switches off about half the units, so with std 1/√n each layer roughly halves the gradient's energy, and 50 layers compound that. With skips each Jacobian is I + (small term), so there's always a direct identity path back and the gradient arrives with a healthy size (≈ 5). That's why ResNets can be hundreds of layers deep (Murphy §13.4.4, §14.3.4).

Expected output:
```
plain ReLU stack: |grad at layer 1| = 3.190e-08
residual stack:   |grad at layer 1| = 5.606e+00
```

</details>